In [12]:
import torch
from transformers import AutoModelForSeq2SeqLM, MBart50TokenizerFast

model = AutoModelForSeq2SeqLM.from_pretrained('facebook/mbart-large-50-many-to-many-mmt', dtype=torch.bfloat16,
                                              attn_implementation='sdpa', device_map='auto')
tokenizer = MBart50TokenizerFast.from_pretrained('facebook/mbart-large-50-many-to-many-mmt')

/Users/pivanovs/.pyenv/versions/mt-proj/lib/python3.12/site-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['final_logits_bias']
  warnings.warn(
Some parameters are on the meta device because they were offloaded to the disk.


In [ ]:
print(model.device)

In [13]:
SOURCE_LANG = 'en_XX'
TARGET_LANG = 'uk_UA'

In [14]:
tokenizer.src_lang = SOURCE_LANG
tokenizer.tgt_lang = TARGET_LANG

In [15]:
import pandas as pd

df = pd.read_csv('../data/translations.tsv', sep='\t')
test_sentences = list(df['source'])

In [16]:
test_sentences

['<neutral> I used to work as a bartender.',
 '<neutral> I am a poor bartender.',
 '<neutral> I am a nice bartender.',
 '<neutral> The bartender called me today.',
 '<female> Anne is a bartender.',
 '<female> Anne is an experienced bartender.',
 '<male> Tom is a bartender.',
 '<male> Tom is a poor bartender.',
 '<neutral> Jessie is a good bartender.',
 '<neutral> Jessie is a poor bartender.',
 '<neutral> I used to work as a paramedic.',
 '<neutral> I am a poor paramedic.',
 '<neutral> I am a nice paramedic.',
 '<neutral> The paramedic called me today.',
 '<female> Anne is a paramedic.',
 '<female> Anne is an experienced paramedic.',
 '<male> Tom is a paramedic.',
 '<male> Tom is a poor paramedic.',
 '<neutral> Jessie is a good paramedic.',
 '<neutral> Jessie is a poor paramedic.',
 '<neutral> I used to work as a examiner.',
 '<neutral> I am a poor examiner.',
 '<neutral> I am a nice examiner.',
 '<neutral> The examiner called me today.',
 '<female> Anne is a examiner.',
 '<female> Anne

In [17]:
from itertools import batched
from tqdm import tqdm


def translate(txt: str | list[str], model, tokenizer) -> list[str]:
    translated = []
    for batch in tqdm(list(batched(txt, n=4))):
        encoded = tokenizer(batch, return_tensors='pt', padding=True).to(model.device)
        generated = model.generate(**encoded, forced_bos_token_id=tokenizer.lang_code_to_id[tokenizer.tgt_lang],
                                   cache_implementation='static')
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        translated.extend(decoded)
    return translated

In [18]:
translated_sentences = translate(test_sentences, model, tokenizer)
translated_sentences

100%|██████████| 203/203 [08:09<00:00,  2.41s/it]


['Я раніше працював барменом.',
 'Я бідний бармен.',
 'Я гарний бармен.',
 'Бармен зателефонував мені сьогодні.',
 'Анна — бармен.',
 'Анна досвідчений бармен.',
 'Том бармен.',
 'Том — бідний бармен.',
 'Джеї - добрий бармен.',
 'Джесси — бідний бармен.',
 'Я раніше працював фельдшером.',
 'Я бідний фельдшер.',
 'Я чудовий фельдшер.',
 'Медсестра зателефонував мені сьогодні.',
 'Анна — фельдшер.',
 'Анна досвідчений фельдшер.',
 'Том — фельдшер.',
 'Том — бідний фельдшер.',
 'Джеї - добрий фельдшер.',
 'Джесси — бідний фельдшер.',
 'Я раніше працював Examinerом.',
 'Я бідний оглядач.',
 'Я чудовий оглядач.',
 'Провізор зателефонував мені сьогодні.',
 'Анна — оглядач.',
 'Анна досвідчений оглядач.',
 'Том — оглядач.',
 'Том — бідний оглядач.',
 'Джесси - добрий оглядач.',
 'Джесси — бідний оглядач.',
 'Я раніше працював механіком.',
 'Я бідний механік.',
 'Я чудовий механік.',
 'Мене сьогодні зателефонував механік.',
 'Анна — механік.',
 'Анна досвідчений механік.',
 'Том — механік.',


In [20]:
df['mbart_hypothesis'] = translated_sentences
df.to_csv('../data/mbart.zeroshot.translations.tsv', sep='\t', index=False)